In [1]:
import pandas as pd 

In [2]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
tags = pd.read_csv("../data/tags.csv")

In [3]:
display(movies.head())
display(ratings.head())
display(tags.head())

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


In [4]:
movies.info()
ratings.info()
tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  87585 non-null  int64
 1   title    87585 non-null  str  
 2   genres   87585 non-null  str  
dtypes: int64(1), str(2)
memory usage: 5.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 2000072 entries, 0 to 2000071
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   userId     int64
 1   movieId    int64
 2   tag        str  
 3   timestamp  int64
dtypes: int64(3), str(1)
memory usage: 82.5 MB


In [5]:
dataset_stats = {
    "Users": ratings['userId'].nunique(),
    "Movies": ratings['movieId'].nunique(),
    "Ratings": len(ratings),
    "Tags": len(tags)
}

pd.DataFrame(
    dataset_stats.items(),
    columns=['Metric', 'Value']
)

,Metric,Value
0,Users,200948
1,Movies,84432
2,Ratings,32000204
3,Tags,2000072


In [6]:
display(movies.isnull().sum())
display(ratings.isnull().sum())
display(tags.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

In [9]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['clean_title'] = movies['title'].str.replace(r'\(\d{4}\)', '', regex=True)

tags = tags.dropna(subset=['tag']).copy()
tags['tag'] = tags['tag'].astype(str)

tag_group = (
    tags.groupby('movieId')['tag']
    .apply(lambda x: ' '.join(x))
    .reset_index()
)

movies = movies.merge(
    tag_group,
    on='movieId',
    how='left'
)

display(movies.head())

,movieId,title,genres,clean_title,tag
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story,children Disney animation children Disney Disn...
1,2,Jumanji (1995),Adventure Children Fantasy,Jumanji,Robin Williams fantasy Robin Williams time tra...
2,3,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men,comedinha de velhinhos engraÃƒÂ§ada comedinha ...
3,4,Waiting to Exhale (1995),Comedy Drama Romance,Waiting to Exhale,characters slurs based on novel or book chick ...
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Fantasy pregnancy remake family Steve Martin s...


In [10]:
movies['tag'] = movies['tag'].fillna('')

movies['metadata'] = (
    movies['clean_title'] + ' ' +
    movies['genres'] + ' ' +
    movies['tag']
)

In [11]:
display(movies[['movieId', 'title', 'metadata']].head())

,movieId,title,metadata
0,1,Toy Story (1995),Toy Story Adventure Animation Children Comedy...
1,2,Jumanji (1995),Jumanji Adventure Children Fantasy Robin Will...
2,3,Grumpier Old Men (1995),Grumpier Old Men Comedy Romance comedinha de ...
3,4,Waiting to Exhale (1995),Waiting to Exhale Comedy Drama Romance charac...
4,5,Father of the Bride Part II (1995),Father of the Bride Part II Comedy Fantasy pr...


In [12]:
movies.to_csv("../data/processed_movies.csv", index=False)